# SafeMaint Qwen3.5-9B Colab Server

This notebook starts the same `ai/qwen_service` FastAPI server used by local Docker. It does not mount Google Drive. Use it for development/demo only; company deployment should run the Qwen service on an internal GPU server. This version exposes the server with ngrok.

In [ ]:
# Runtime configuration
SAFE_MAINT_REPO_URL = ""  # optional: https://github.com/your-org/your-repo.git
PROJECT_ROOT = "/content/safemaint"

QWEN_BASE_MODEL = "Qwen/Qwen3.5-9B"
QWEN_LORA_ADAPTER = "/content/qwen_adapter"
QWEN_API_KEY = "change-this-shared-demo-token"
QWEN_LOAD_IN_4BIT = "true"
QWEN_MAX_NEW_TOKENS = "768"
QWEN_CLASSIFY_MAX_NEW_TOKENS = "96"

NGROK_AUTH_TOKEN = ""  # required: https://dashboard.ngrok.com/get-started/your-authtoken

# If blank, the notebook will ask you to upload an adapter zip.
ADAPTER_ZIP_URL = ""


In [ ]:
# Install runtime packages. Colab normally already has torch with CUDA.
!pip -q install fastapi uvicorn transformers accelerate peft bitsandbytes safetensors pydantic pyngrok


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

project_root = Path(PROJECT_ROOT)
if project_root.exists():
    print(f"Project already exists: {project_root}")
elif SAFE_MAINT_REPO_URL:
    subprocess.run(["git", "clone", SAFE_MAINT_REPO_URL, str(project_root)], check=True)
else:
    from google.colab import files
    print("Upload a project zip that contains ai/qwen_service/.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No project zip uploaded.")
    archive = Path(next(iter(uploaded.keys()))).resolve()
    unpack_dir = Path("/content/safemaint_upload")
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    shutil.unpack_archive(str(archive), str(unpack_dir))
    # Accept zips created on Windows too. Some zip tools store paths like
    # ai\\qwen_service\\main.py, which Linux treats as flat filenames.
    for path in list(unpack_dir.rglob("*")):
        if "\\" in path.name:
            target = unpack_dir / Path(path.name.replace("\\", "/"))
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(path), str(target))
    candidates = [p for p in unpack_dir.rglob("qwen_service") if (p / "main.py").exists()]
    if not candidates:
        raise RuntimeError("Could not find ai/qwen_service in uploaded zip.")
    source_root = candidates[0].parents[1]
    shutil.copytree(source_root, project_root)

qwen_service_dir = project_root / "ai" / "qwen_service"
if not (qwen_service_dir / "main.py").exists():
    raise RuntimeError(f"qwen_service not found: {qwen_service_dir}")
print(f"Using project root: {project_root}")


In [ ]:
# Prepare LoRA adapter without Google Drive.
from pathlib import Path
import shutil
import subprocess

adapter_dir = Path(QWEN_LORA_ADAPTER)
if adapter_dir.exists() and (adapter_dir / "adapter_config.json").exists():
    print(f"Adapter already exists: {adapter_dir}")
else:
    if adapter_dir.exists():
        shutil.rmtree(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    if ADAPTER_ZIP_URL:
        zip_path = Path("/content/qwen_adapter.zip")
        subprocess.run(["wget", "-O", str(zip_path), ADAPTER_ZIP_URL], check=True)
    else:
        from google.colab import files
        print("Upload the LoRA adapter zip. It must contain adapter_config.json and adapter_model.safetensors.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No adapter zip uploaded.")
        zip_path = Path(next(iter(uploaded.keys()))).resolve()
    unpack_dir = Path("/content/qwen_adapter_unpacked")
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    shutil.unpack_archive(str(zip_path), str(unpack_dir))
    configs = list(unpack_dir.rglob("adapter_config.json"))
    if not configs:
        raise RuntimeError("adapter_config.json not found in adapter zip.")
    found_adapter_dir = configs[0].parent
    for item in found_adapter_dir.iterdir():
        target = adapter_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
print(f"Using adapter dir: {adapter_dir}")


In [ ]:
# Start the Qwen FastAPI server.
import os
import subprocess
import time

os.environ["QWEN_BASE_MODEL"] = QWEN_BASE_MODEL
os.environ["QWEN_LORA_ADAPTER"] = QWEN_LORA_ADAPTER
os.environ["QWEN_API_KEY"] = QWEN_API_KEY
os.environ["QWEN_DEVICE"] = "cuda"
os.environ["QWEN_LOAD_IN_4BIT"] = QWEN_LOAD_IN_4BIT
os.environ["QWEN_MAX_NEW_TOKENS"] = QWEN_MAX_NEW_TOKENS
os.environ["QWEN_CLASSIFY_MAX_NEW_TOKENS"] = QWEN_CLASSIFY_MAX_NEW_TOKENS
os.environ.setdefault("HF_HOME", "/content/hf_cache")

subprocess.run("pkill -f 'uvicorn qwen_service.main:app' || true", shell=True)
server_log = open("/content/qwen_server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "qwen_service.main:app", "--host", "127.0.0.1", "--port", "8020"],
    cwd=str(project_root / "ai"),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
print("Qwen service process id:", server.pid)
!curl -i http://127.0.0.1:8020/health/live
!tail -n 40 /content/qwen_server.log


In [ ]:
# Expose the local FastAPI server with ngrok.
if not NGROK_AUTH_TOKEN:
    raise RuntimeError("Set NGROK_AUTH_TOKEN in the first configuration cell.")

from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8020, "http").public_url

print("ngrok Public URL:", public_url)
!curl -i http://127.0.0.1:8020/health/live
import urllib.request
with urllib.request.urlopen(public_url + "/health/live", timeout=30) as response:
    print("Public health status:", response.status)
    print(response.read().decode("utf-8")[:500])

print("\nSet this in each teammate .env:")
print("QWEN_ENABLED=true")
print("QWEN_PROVIDER=colab")
print(f"QWEN_SERVICE_URL={public_url}")
print(f"QWEN_API_KEY={QWEN_API_KEY}")
print("QWEN_ALLOW_COMPANY_CONTEXT=true")
